In [17]:
import os


os.makedirs("trading_bot/bot", exist_ok=True)

print("Folders created successfully on your computer!")

Folders created successfully on your computer!


In [5]:
%%writefile trading_bot/requirements.txt
python-binance

Writing trading_bot/requirements.txt


In [6]:
%%writefile trading_bot/bot/__init__.py
# This file is intentionally left blank.

Writing trading_bot/bot/__init__.py


In [7]:
%%writefile trading_bot/bot/logging_config.py
import logging

def setup_logging():
    logging.basicConfig(
        filename='trading.log',
        level=logging.INFO,
        format='%(asctime)s - %(levelname)s - %(message)s'
    )
    console = logging.StreamHandler()
    console.setLevel(logging.INFO)
    logging.getLogger('').addHandler(console)

Writing trading_bot/bot/logging_config.py


In [8]:
%%writefile trading_bot/bot/client.py
import logging
from binance.client import Client

class BinanceTestnetClient:
    def __init__(self, api_key, api_secret):
        if not api_key or not api_secret:
            raise ValueError("API Key and Secret Key cannot be empty!")
            
        logging.info("Initializing Binance Testnet Client...")
        self.client = Client(api_key, api_secret, testnet=True)
        
    def get_raw_client(self):
        return self.client

Writing trading_bot/bot/client.py


In [9]:
%%writefile trading_bot/bot/orders.py
import logging

def execute_futures_order(client_wrapper, symbol, side, order_type, quantity, price=None):
    client = client_wrapper.get_raw_client()
    symbol = symbol.upper()
    side = side.upper()
    order_type = order_type.upper()

    if side not in ['BUY', 'SELL']:
        raise ValueError(f"Invalid side: {side}. Must be BUY or SELL.")
    if order_type not in ['MARKET', 'LIMIT']:
        raise ValueError(f"Invalid order type: {order_type}. Must be MARKET or LIMIT.")
    if order_type == 'LIMIT' and not price:
        raise ValueError("A price is required for LIMIT orders!")

    logging.info(f"[REQUEST SUMMARY] Placing {side} {order_type} order for {quantity} {symbol}...")

    try:
        order_params = {
            'symbol': symbol,
            'side': side,
            'type': order_type,
            'quantity': float(quantity)
        }
        
        if order_type == 'LIMIT':
            order_params['price'] = str(price)
            order_params['timeInForce'] = 'GTC'

        response = client.futures_create_order(**order_params)
        
        logging.info("[SUCCESS] Order executed successfully by testnet server.")
        
        summary = {
            "orderId": response.get("orderId"),
            "status": response.get("status"),
            "executedQty": response.get("executedQty"),
            "avgPrice": response.get("avgPrice", "N/A")
        }
        return True, summary

    except Exception as e:
        error_msg = f"[FAILURE] Binance API rejected the order. Reason: {str(e)}"
        logging.error(error_msg)
        return False, str(e)

Writing trading_bot/bot/orders.py


In [14]:
%%writefile trading_bot/cli.py
import argparse
import sys
from bot.logging_config import setup_logging
from bot.client import BinanceTestnetClient
from bot.orders import execute_futures_order


API_KEY = "My_API_Key"
API_SECRET = "My_API_Key_Secret_Key"

def main():
    setup_logging()
    
    parser = argparse.ArgumentParser(description="Simplified Binance Futures Testnet Trading Bot CLI")
    parser.add_argument("--symbol", required=True, help="e.g. BTCUSDT")
    parser.add_argument("--side", required=True, help="BUY or SELL")
    parser.add_argument("--type", required=True, help="MARKET or LIMIT")
    parser.add_argument("--quantity", required=True, type=float, help="Amount to trade")
    parser.add_argument("--price", type=float, help="Price (Required ONLY for LIMIT orders)")
    
    args = parser.parse_args()

    try:
        bot_client = BinanceTestnetClient(API_KEY, API_SECRET)
        success, result = execute_futures_order(
            client_wrapper=bot_client, symbol=args.symbol, side=args.side, 
            order_type=args.type, quantity=args.quantity, price=args.price
        )
        
        print("\n" + "="*40)
        if success:
            print("[+] ORDER CONFIRMED")
            for key, val in result.items():
                print(f" > {key}: {val}")
        else:
            print(f"[-] ORDER FAILED\nReason: {result}")
        print("="*40 + "\n")

    except Exception as e:
        print(f"\n[!] Error: {str(e)}\n")

if __name__ == "__main__":
    main()

Overwriting trading_bot/cli.py


In [11]:
!pip install -r trading_bot/requirements.txt

   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ----------------------- ---------------- 1.0/1.8 MB 3.9 MB/s eta 0:00:01
   ---------------------------------- ----- 1.6/1.8 MB 3.8 MB/s eta 0:00:01
   ---------------------------------------- 1.8/1.8 MB 2.8 MB/s  0:00:00

   ---------- ----------------------------- 1/4 [pycryptodome]
   ---------- ----------------------------- 1/4 [pycryptodome]
   ---------- ----------------------------- 1/4 [pycryptodome]
   ---------- ----------------------------- 1/4 [pycryptodome]
   ---------- ----------------------------- 1/4 [pycryptodome]
   ---------- ----------------------------- 1/4 [pycryptodome]
   ---------- ----------------------------- 1/4 [pycryptodome]
   ---------- ----------------------------- 1/4 [pycryptodome]
   ---------- ----------------------------- 1/4 [pycryptodome]
   ---------- ----------------------------- 1/4 [pycryptodome]
   ---------- ----------------------------- 1/4 [pycryptodome]
   ---------

In [15]:
!python trading_bot/cli.py --symbol BTCUSDT --side BUY --type MARKET --quantity 0.01


[+] ORDER CONFIRMED
 > orderId: 13854735016
 > status: NEW
 > executedQty: 0.0000
 > avgPrice: 0.00



Initializing Binance Testnet Client...
[REQUEST SUMMARY] Placing BUY MARKET order for 0.01 BTCUSDT...
[SUCCESS] Order executed successfully by testnet server.


In [16]:
!python trading_bot/cli.py --symbol BTCUSDT --side BUY --type LIMIT --quantity 0.01 --price 55000


[+] ORDER CONFIRMED
 > orderId: 13854969284
 > status: NEW
 > executedQty: 0.0000
 > avgPrice: 0.00



Initializing Binance Testnet Client...
[REQUEST SUMMARY] Placing BUY LIMIT order for 0.01 BTCUSDT...
[SUCCESS] Order executed successfully by testnet server.
